# getting extents for PNG from LCCS BCEs
- load in PNG province files (prepared by Chloe)
- get LCCS for extent (through geobox)
- get seagrass extent (through geobox)
- could pixels of BCEs and put into table

In [1]:
import sys
import numpy as np
import geopandas as gpd

import fiona
from shapely.geometry import shape

import rioxarray
from rasterio.features import geometry_mask

import datacube
dc = datacube.Datacube(app="extents")
from datacube.utils.aws import configure_s3_access
from dea_tools.datahandling import load_reproject

# 8.4.25 - Matt Paget work around for error CPLE_HttpResponseError: CURL error: Failed to connect to easi-caching-proxy.caching-proxy port 80 after 0 ms: Couldn't connect to server
sys.path.insert(1, "/home/jovyan/code/easi-notebooks/")
from easi_tools.notebook_utils import unset_cachingproxy


In [2]:
# Access AWS "requester-pays" buckets
# This is necessary for reading data from most third-party AWS S3 buckets such as for Landsat and Sentinel-2
configure_s3_access(aws_unsigned=False, requester_pays=True);

In [3]:
# load in PNG province buffered by TSZ (territorial sea zone, 22km buffered to sea to inlcude seagrass)
# province_file = '../data/PNG_province_TSZbuffered_EPSG32755/Gulf_EPSG32755.shp'
province_file = '../data/PNG_province_TSZbuffered_EPSG32755/Gulf_prov_TSZ.shp'
province_df = gpd.read_file(province_file)
province_df

,Id,geometry
0,0,"POLYGON ((57728.566 9249544.297, 57753.959 924..."


In [4]:
province_df.explore()

In [5]:
# This defines the function that converts a linear vector file into a string of x,y coordinates

def geom_query(geom, geom_crs='EPSG:32755'):
    """
    Create datacube query snippet for geometry
    """
    return {
        'x': (geom.bounds[0], geom.bounds[2]),
        'y': (geom.bounds[1], geom.bounds[3]),
        'crs': geom_crs
    }

def warp_geometry(geom, crs_crs, dst_crs):
    """
    warp geometry from crs_crs to dst_crs
    """
    return shapely.geometry.shape(rasterio.warp.transform_geom(crs_crs, dst_crs, shapely.geometry.mapping(geom)))

In [6]:
# use fiona module to open the shape file
province = fiona.open(province_file)

geom_ = shape(province[0]['geometry'])
geom_query_ = geom_query(geom=geom_)

crs = "EPSG:32755"
res = (30, -30)

query =({'output_crs':crs,
         'resolution':res})

query.update(geom_query(geom=geom_, geom_crs=province.crs_wkt))

In [7]:
# load in GLO data 
with unset_cachingproxy():
    GLO30 = dc.load(product='copernicus_dem_30', resampling='bilinear', time = ('2022-01-01', '2022-12-31'), **query)

In [8]:
# load LCCS band 5 for BCEs
# LCCS data EPSG:32755 30m pixel
LCCS_path = '/home/jovyan/code/livingearth_png/notebooks/png_lccs_classification_v0_2_data_merged.tif'
LCCS_load = load_reproject(path=LCCS_path, how=GLO30.odc.geobox).load()
LCCS_load = LCCS_load.rename('lccs')

/env/lib/python3.12/site-packages/rasterio/warp.py:344: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  _reproject(


In [9]:
# Clip the DataArray to the GeoDataFrame
lccs_clipped = LCCS_load.rio.clip(province_df.geometry, province_df.crs, drop=False, invert=False)

# Mask out values outside polygon with NaNs (if not already)
lccs_clipped = lccs_clipped.where(~lccs_clipped.isnull(), other=np.nan)

lccs_clipped

<xarray.DataArray 'lccs' (band: 5, y: 7664, x: 13710)> Size: 2GB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]], dtype=float32)
Coordinates:
  * band         (band) int64 40B 1 2 3 4 5
  * y            (y) float64 61kB 9.033e+06 9.033e+06 ... 9.263e+06 9.263e+06
  * x            (x) float64 110kB 4.641e+05 4.64e+05 ... 5.282e+04 5.279e+04
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:  Area
    scale_factor:   1.0
    add_offset:     0.0

In [10]:
# Assuming lccs is your DataArray
unique_values, counts = np.unique(lccs_clipped.values, return_counts=True)

# Multiply counts by 625 to get area (in m²)
areas = counts * 0.0009

# Print results
for value, count, area in zip(unique_values, counts, areas):
    print(f"Value: {value}, Count: {count}, Area (km²): {area}")

Value: 0.0, Count: 55305454, Area (km²): 49774.908599999995
Value: 1.0, Count: 2254052, Area (km²): 2028.6468
Value: 2.0, Count: 2672729, Area (km²): 2405.4561
Value: 3.0, Count: 66473, Area (km²): 59.8257
Value: 10.0, Count: 26795318, Area (km²): 24115.7862
Value: 20.0, Count: 12787087, Area (km²): 11508.3783
Value: 100.0, Count: 31762542, Area (km²): 28586.2878
Value: 112.0, Count: 26769288, Area (km²): 24092.3592
Value: 124.0, Count: 4993254, Area (km²): 4493.9286
Value: 200.0, Count: 8232401, Area (km²): 7409.1609
Value: 215.0, Count: 215, Area (km²): 0.1935
Value: 216.0, Count: 25815, Area (km²): 23.2335
Value: 220.0, Count: 7793833, Area (km²): 7014.4497
Value: 1121.0, Count: 50617734, Area (km²): 45555.9606
Value: 1122.0, Count: 2920842, Area (km²): 2628.7578
Value: 1241.0, Count: 4926781, Area (km²): 4434.1029
Value: 1242.0, Count: 66473, Area (km²): 59.8257
Value: 2000.0, Count: 412538, Area (km²): 371.2842
Value: 2150.0, Count: 430, Area (km²): 0.387
Value: 2160.0, Count: 516

In [11]:
# LCCS_load.odc.write_cog('LCCS_load_Gulf_EPSG32755.tif', overwrite=True)